# Classification d'émotions dans la parole avec un modèle Bayésien naïf.

Dans ce travail, vous allez travailler sur un corpus de parole où les locuteurs actent des émotions sur un texte prédéfini. Ce corpus s'appelle AIBO et il a été utilisé pour une compétition internationale sur la détection des émotions (Interspeech 2009).
L'article de référence est celui-ci:
*A. Batliner, C. Hacker, S. Steidl, E. Nöth, S. D’Arcy, M. Russell, and M. Wong, “«you stupid tin box» - children interacting with the aibo robot: A cross-linguistic emotional speech corpus,” in LREC, Lisbon, Portugal, 2004, pp. 171–174.* accessible ici http://www.lrec-conf.org/proceedings/lrec2004/pdf/317.pdf

Dans ce corpus, 51 enfants ont été enregistrés dans 2 écoles allemandes (notées MONT et OHM). Ils devaient interagir avec un chien robot (appellé Aibo). Le robot était en fait piloté par un opérateur et désobéissait, ce qui pouvait induire des émotions spontanées et variées chez les enfants.

5 émotions ont été annotées dans la parole: colère, empathie, neutre, positive et le reste. Pour simplifier, nous avons réduit notre problème a une classification binaire qui suit une dimension affective, la valence :
* NEG: colère et empathie
* IDL: neutre, positive et reste.

Dans notre protocole, les enregistrements d'une école seront utilisés pour l'apprentissage du modèle (Mont), tandis que les enregistrements de l'autre école seront utilisés pour évaluer le modèle (Ohm). Ce protocole, permet d'évaluer la capacité de généralisation du modèle à des locuteurs inconnus.

In [ ]:
from pathlib import Path
import pandas as pd
import scipy.io.wavfile as wav
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

## 1) Feature extraction (à lire et exécuter que pour la partie 4)

Vous n'avez rien à faire dans cette section. Je vous la laisse pour comprendre comment sont construits les différents fichiers que vous aurez à traiter par la suite.

Pour construire un modèle à partir d'une base de donnée audio, il faut extraire des descripteurs du signal de parole. Généralement ces descripteurs sont issus de la théorie du traitement numérique du signal et du modèle source-filtre de la parole.

Nous considérons ici, l'extraction des descripteurs comme une boîte noire qui prend en entrée un signal audio (fichier wav) et retourne un vecteur de caractéristiques pour l'ensemble du segment. La taille du vecteur ne dépend pas de la durée du signal audio.

La boîte noire que nous allons utiliser est celle d'OpenSmile (https://audeering.github.io/opensmile-python/). Le contenu des vecteurs de caractéristiques varie suivant la configuration choisie. Une des configurations les plus utilisées s'appelle eGeMAPS, c'est celle que nous utiliserons. Elle permet de représenter un segment de parole émotionnelle sous forme d'un vecteur de 88 valeurs continues.

In [ ]:
import opensmile
smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv02,
    feature_level=opensmile.FeatureLevel.Functionals,
)
# affichage des noms des descripteurs audio qui sont extraits pour chaque fichier audio.
feat = smile.feature_names
print(','.join(feat))

In [ ]:
# à ne pas exécuter sauf question 4
def write_extract_features(corpus_dir, spk_list, csvfile):
    """
        Cette fonction crée un fichier contenant l'ensemble des descripteurs obtenus pour chaque fichier audio,
        ainsi que la classe d'émotion correspondante: Name,desc1,desc2,...,desc88,Emo
        - corpus_dir: chemin vers les fichiers audio
        - spk_list: liste des locuteurs considérés pour la création du fichier
        - csvfile: nom du fichier
    """
    header = 'Name,'+','.join(feat)+',Emo\n'
    with open(csvfile, 'w') as f:
        f.write(header)
        for file in Path(corpus_dir).glob('*.wav'):
            
            tab = str(file).split('/')[-1]
            spk = tab[:2]
            if spk not in spk_list:
                continue
            text = tab[2:5]
            emo = emo2cl[tab[5]]
            if emo == 'na':
                continue
            sr, y = wav.read(corpus_dir + tab)
            x = smile.process_signal(y,sr)
            f.write(tab + ',')
            f.write(','.join(str(x.values[0][i]) for i in range(88)))
            f.write(',' + emo + '\n')
            
def map_label_wav(corpus_dir, pd_label):
    pd_wav = pd.DataFrame(columns=['name', 'wavfile'])
    for i, file in enumerate(Path(corpus_dir).glob('*.wav')):
        name = str(file).split('/')[-1]
        pd_wav.loc[len(pd_wav)] = [name[:-4], file]

        pd_wav.set_index('name')
        pd_full = pd_wav.merge(pd_label, how='inner', on='name')
    return pd_full

In [ ]:
# à ne pas exécuter sauf question 4
aibo_path = '/Users/tahon/Documents/Recherche/Corpus/AIBO/'

aibo_train_dir = aibo_path + 'wav/AIBO-M/' 
aibo_test_dir  = aibo_path + 'wav/AIBO-O/'
aibo_labels    = aibo_path + 'wav/chunk_labels_2cl_corpus.txt' # à changer si vous voulez 5 classes d'émotions

aibo = pd.read_csv(aibo_labels, sep=' ',header=None, names=['name', 'emo2cl', 'val'])
del aibo['val']
aibo.set_index('name')
aibo

In [ ]:
# à ne pas exécuter sauf question 4
# extraction des features sur le train
aibo_train = map_label_wav(aibo_train_dir, aibo)
# write_extract_features(aibo_train, 'aibo_train.csv')
aibo_train.loc[0]

In [ ]:
# à ne pas exécuter sauf question 4
# extraction des features sur le test
aibo_test = map_label_wav(aibo_test_dir, aibo)
# write_extract_features(aibo_test, 'aibo_test.csv')
aibo_test.loc[0]

--------------------------
A partir de maintenant, nous pouvons extraire l'ensemble des descripteurs des ensembles d'apprentissage et de test fichier par fichier.

Si vous souhaitez modifier la répartition des classes émotionnelles, et/ou faire une classification à plusieurs classes, vous devrez regénérer ces fichiers. Mais pour la partie suivante, vous pouvez récupérer les fichiers directement sur le dépôt UMTICE.

### Extract train set features

In [ ]:
#spk_train = ['03', '08', '09', '10', '11', '12']
#write_extract_features(corpus_dir, spk_train, 'emodb_train.csv')

### Extract test features

In [ ]:
#spk_test = ['13', '14', '15', '16']
#write_extract_features(corpus_dir, spk_test, 'emodb_test.csv')

## 2) Analyse des données (5 pts)

Avant de choisir quel modèle vous souhaitez mettre en place, il va être nécessaire d'étudier les données pour voir comment elles sont organisées. Dans un premier temps on évaluera si tous les descripteurs sont pertinents : est-ce que certains sont très fortement corrélés ? d'autres ont peut-être une variance presque nulle, etc... Dans un second temps on cherchera à savoir la loi de probabilité qui est la plus adaptée pour décrire la distribution des données. 

### Chargement des données d'apprentissage et de test

Pour charger les données on utilisera la librairie pandas. Si vous êtes plus à l'aise avec numpy, c'est possible de convertir les DataFrame de pandas en numpy avec la méthodes `to_numpy()`

Vous pouvez charger directement les données fournies au format CSV avec le code ci-dessous. Ainsi vous n'avez pas besoin d'effectuer le calcul des descripteurs et d'écrire le CSV, cette étape (partie 1) est déjà faite pour ce TP.

In [ ]:
# on récupère deux panda dataframe contenant les noms des fichiers, les descripteurs associés et l'émotion réelle.
train_data = pd.read_csv('aibo_train.csv')
test_data = pd.read_csv('aibo_test.csv')

# to get column names
feat_names = train_data.columns.values

**Q1** À partir des données d'apprentissage uniquement, tracer la distribution de quelques descripteurs. Donner le type des descripteurs (continu, discret), et la loi (ou les lois) qui permet de décrire au mieux ces données. Vous justifierez au maximum votre réponse. Il est possible que certains descripteurs soient inutiles, vous prendrez soin de nettoyer vos données avant de passer à la suite.

In [ ]:

print("Liste des descripteurs originals:")
print(train_data.columns.tolist())
print(f"Il y a {len(train_data.columns)} descripteurs avant le nettoyage.")

# On sépare x les indices et y la réponse
# On enlève 'Name' (inutile) et 'Emo' (réponse)

x_train = train_data.drop(columns=['Name', 'Emo'])
y_train = train_data['Emo']
# pareil pour le Test
x_test = test_data.drop(columns=['Name', 'Emo'])
y_test = test_data['Emo']

# nettoyage:  On enlève les colonnes qui ne servent à rien car elles valent toujours 0
# On cherche les colonnes où l'écart-type (std) est 0
colonnes_inutiles = x_train.columns[x_train.std() == 0]
x_train = x_train.drop(columns=colonnes_inutiles)
x_test = x_test.drop(columns=colonnes_inutiles)

print(f" Après nettoyage : {len(colonnes_inutiles)} colonnes inutiles supprimées.")

In [ ]:
#Affichage les descripteurs apres le nettoyage de descripteurs inutils
print("Liste des descripteurs restants :")
print(x_train.columns.tolist())
print(f"Il reste {len(x_train.columns)} descripteurs après le nettoyage.")

In [ ]:
### TO COMPLETE

# On choisit colonnes restantes quelconques pour tracer la distribution
features_a_voir = ['alphaRatioUV_sma3nz_amean', 'mfcc4_sma3_amean', 'mfcc1_sma3_amean', 'mfcc2_sma3_amean']

plt.figure(figsize=(25, 5))

for i, col in enumerate(features_a_voir):
    # on vérifie que la colonne existe avant de tracer
    if col in x_train.columns:
        plt.subplot(1, 4, i+1)
        plt.hist(x_train[col], bins=30, color='skyblue', edgecolor='black', density=True)
        plt.title(f"Distribution : {col}")
        plt.xlabel("Valeur")
        plt.ylabel("Fréquence")

plt.tight_layout()
plt.show()

**Justification de Q1** Le type de descripteurs est continue, en effet, en regardant dans le fichier .csv. Il n'existe que les valeurs réelles flottantes car elles varient progressivement. 
Ces données suivent la loi Normale N(moyenne, ecart-type) car avec la destribution de quelques descripteurs, on les retrouve les bosse et leurs côtes redescendent, ce qui est sous la forme de cloche.
On effectue également le nettoyage de descripteurs qui n'apportent aucune information pour la suite et le dessin de destribution, comme les colonnes à variance null ou qui sont constantes. En plus, on enlève pour l'instant les colonnes Name et Emo, qui semblent d'être inutiles dans cette question.

**Q2** Maintenant que vos données sont un peu plus propres, vous pouvez estimer à quel point les descripteurs sont indépendants. Proposer pour cela une méthodologie d'analyse en justifiant vos choix. En déduire certaines propriétés des descripteurs par rapport à nos données d'apprentissage.

In [ ]:
### TO COMPLETE
# Calcul de la matrice de corrélation pour interpreter si les descripteurs sont indépendant ou pas
correlation = x_train.corr()

plt.figure(figsize=(10, 8))
# afficher des matrices comme des images
image = plt.imshow(correlation, cmap='coolwarm', vmin=-1, vmax=1)

# ajout de la légende
plt.colorbar(image, label="Corrélation")
plt.title("Matrice de Corrélation des variables")
plt.show()

**Justification de Q2** Pour savoir à quel point les descripteurs sont indépendants, on cherche à calculer la matrice de covariance comme dans cours, qui permet d'observer si les descripteur sont liés entre eux ou pas. Vu que les descripteurs de train_data ont des données sous différente forme de chiffres, certains sont trop petits, négatifs et beaucoup grands de l'un à l'autre. ce qui ne rend pas clair au niveau affichage des liens entre chacun de des descripteurs, car il y a un problème echelle, lorsque les descripteur ont grand écart-type écrasent celui de petit écart-type. On ne peut pas utiliser la matrice de covariance pour faire analyse. Il suffit donc à vérifier si la matrice de corrélation est diagonale. Elle normalise la matrice de covariance entre -1 et 1. Ce dernier rend plus visible et facilement à analyser.

D'après le graphique, on retrouve que la matrice de corrélation n'est pas diagonale, car il y a toujours des cases de corrélation fortes et faibles hors de diagonale, ceux qui n'sont pas complètement nuls(couleur grise). Par conséquence, la matrice de la matrice de covariance est aussi non diagonale alors les des descripteurs ne sont pas indépendants.

En fin, on peut déduire que les descripteurs sont des variable continues et qui suivent la loi normale. Mais ils ne sont pas indépendants. De plus, leurs échelles de valeurs ne sont pas homogènes, ce qui a besoin de les normaliser.

## 3) Construction d'un modèle Bayesien Naïf (6 pts)

Dans cette seconde partie vous devez implémenter un modèle Bayésien Naïf. Les paramètres du modèles, c'est-à-dire les paramètres correspondant à la loi de probabilité que vous allez utilisée, seront obtenus à partir des données d'apprentissage.

Les données de test serviront à évaluer la qualité en termes de performance de votre modèle. Vous ne devez en AUCUN cas les utiliser pour déterminer les paramètres du modèle.


**Q3** A partir des éléments de code donnés ci-dessous, calculer la probabilité a priori pour chacune des classes.

In [ ]:
train_data_clean = pd.concat([x_train, y_train], axis=1)
print(train_data_clean.groupby('Emo').size())
### TO COMPLETE

# On utilise y_train qui contient les étiquettes 'Emo'
# Calcul des effectifs -> le nombre d'exemples par classes
counts = y_train.value_counts()

# probabilités = Effectif / Total
# normalize=True fait la division automatiquement
priors = y_train.value_counts(normalize=True)

print("Probabilités A Priori")
print(priors)

# vérifier que la somme fait bien 1
print(f"\nSomme des probabilités : {priors.sum()}")


**Justification de Q3** On calcule les probabilités a priori de chque classe comme le calcule de fréquence d'apparition. P =eff/total. On constate que les classes ne sont pas équilibres car la classe IDL est de 70%, elle a beaucoup chance d'être un enregistrement est de cette classe que celle NEG.

**Q4** Pour chaque descripteur déterminer les paramètres de loi à partir des segments émotionnels de la base d'apprentissage. On rappelle que dans le cas Bernoulli, il faut déterminer la probabilité de succès de l'événement. Dans le cas multinomial, il faut déterminer la moyenne et l'écart-type de la densité de probabilité. 

**Q5** Rappeler l'hypothèse naïve. Qu'en pensez-vous par rapport à l'analyse des données faites dans la partie 2 ?

**Q6** Construire une fonction qui calcule la vraisemblance d'un segment de test d'appartenir à la classe IDL ou NEG. Elle prendra en entrée les données d'apprentissage et un segment de test.

**Q7** Pour un segment de test dont on ne connait pas la classe émotionnelle, déterminer le maximum de vraisemblance. 

In [ ]:
#Pour Q4:
# Calcul des paramètres de la Loi Normale
# On reprend les données de train_data_clean qui contiennent la colonne 'Emo'

# Calcul des moyennes pour chaque émotion
# On groupe par 'Emo' et on calcule la moyenne de toutes les colonnes
moyennes_par_emotion = train_data_clean.groupby('Emo').mean()

# Calcul des Écarts-types pour chaque émotion
ecarts_types_par_emotion = train_data_clean.groupby('Emo').std()

# Affichage des résultats
with pd.option_context('display.max_columns', None):
    print("Moyennes\n")
    print(moyennes_par_emotion)

    print("\nÉcarts-types\n")
    print(ecarts_types_par_emotion)


**Justification de Q4** Par rapport aux questions précedentes, les descripteurs se modèlisent par la loi Normale. On va calculer alors leurs moyennes et leur écart-types. Ces resultats représentent la moyenne et la dispersion de chaque émotion dans chaque descripteurs.

**Réponse de Q5** Hypthède de Naïf est tous les descripteurs sont indépendants de l'un à l'autre en sachant la classe emotion.

Par contre, à la partie 2 avec la matrice de corrélation trouvée montre que les descripteurs sont liés, donc les descripteurs sont dépendants. L'hypothèse devient faux.

In [ ]:
#Pour Q6:

def calculer_vraisemblance(donnees_apprentissage, segment_test):
    
    # Estimation des paramètres sur les données fournies
    data = np.array(donnees_apprentissage)
    # Moyenne
    mu = np.mean(data, axis=0)    
    # Variance
    var = np.var(data, axis=0)     
    
    # On remplace les variances nulles pour éviter la division par 0
    var[var < 1e-9] = 1e-9
    
    # Calcul de la Vraisemblance, formule en cours mais intègré avec logarithme
    epsilon = 1e-9
    x = np.array(segment_test)
    
    # Terme 1
    term1 = -0.5 * np.sum(np.log(2 * np.pi * var + epsilon))
    
    # Terme 2
    # Si x est une matrice (plusieurs tests), on somme sur les colonnes (axis=1)
    if x.ndim > 1:
        diff = x - mu
        term2 = -0.5 * np.sum((diff ** 2) / (var + epsilon), axis=1)
        return term1 + term2
    else:
        # Si x est un seul vecteur
        diff = x - mu
        term2 = -0.5 * np.sum((diff ** 2) / (var + epsilon))
        return term1 + term2

**Justification de Q6** On choissit d'intégrer la notion de logarithme pour éviter les erreurs d'arrondissment comme e⁻¹⁰⁰ est un peu près 0. 

In [ ]:
#Pour Q7:

# On convertit en Numpy
x_train_np = x_train.values
x_test_np = x_test.values
y_train_np = y_train.values
y_test_np = y_test.values

# On sépare les données d'apprentissage par classe
train_IDL = x_train_np[y_train_np == 'IDL']
train_NEG = x_train_np[y_train_np == 'NEG']

#calcul de vraisemblance de chque classe de segment test
scores_idl = calculer_vraisemblance(train_IDL, x_test_np)
scores_neg = calculer_vraisemblance(train_NEG, x_test_np)

# calcule de maximum de vraisemblance
predictions_test = np.where(scores_idl > scores_neg, 'IDL', 'NEG')

print(predictions_test)


**Q8** A partir de la vraisemblance et de la probabilité a priori, calculer la probabilité a posteri que le segment de test à partienne à une classe ou l'autre. Comparer votre résultat avec l'émotion réelle telle qu'annotée dans le corpus.

In [ ]:
# Calcul des Priors
unique, counts = np.unique(y_train_np, return_counts=True)
p_idl = counts[0] / len(y_train_np)
p_neg = counts[1] / len(y_train_np)

# D'apres, la formule de probabilité a posteri dans cours et on la met avec logarithme
# Calcul du Numérateur
# log P(x|c) + log P(c)
log_num_idl = scores_idl + np.log(p_idl)
log_num_neg = scores_neg + np.log(p_neg)

# ---------------------------------------------------------
# ajout de P(x) (Normalisation)
# P(x) = Somme des numérateurs
# log P(x) = log( exp(log_num_idl) + exp(log_num_neg) )
# ---------------------------------------------------------
log_evidence = np.logaddexp(log_num_idl, log_num_neg)

# Calcul de log probabilité a posteri normalisé
# log P(c|x) = Numérateur - Log P(x)
log_prob_idl = log_num_idl - log_evidence
log_prob_neg = log_num_neg - log_evidence

# On repasse en exponentielle pour avoir un chiffre entre 0 et 1
prob_idl = np.exp(log_prob_idl)
prob_neg = np.exp(log_prob_neg)

# Décision
# On vérifie si la probabilité d'être IDL est > 0.5 (50%)
predictions = np.where(prob_idl > 0.5, 'IDL', 'NEG')


In [ ]:

# Création d'un tableau comparatif
comparaison = pd.DataFrame({
    # L'annotation du corpus reel
    'Réalité': y_test_np,           
    'Prédiction': predictions,   # Votre calcul MAP
})

# Ajout d'une colonne  de resulatat de comparaison entre prediction et realité
comparaison['Resultat'] = (comparaison['Réalité'] == comparaison['Prédiction'])

# Affichage des 10 premiers segments pour "voir" la comparaison
print("Comparaison Segment par Segment")
print(comparaison.head(10))

# Resultat
nb_erreurs = len(comparaison) - comparaison['Resultat'].sum()
print(f"\nNombre total d'erreurs : {nb_erreurs} sur {len(comparaison)} segments.")

# test sur segment 0
print(f"Test sur Segment 0 :")
print(f"Probabilité d'être IDL : {prob_idl[0]*100:.2f}%")
print(f"Probabilité d'être NEG : {prob_neg[0]*100:.2f}%")
print(f"Décision : {predictions[0]}")

**Justification de Q8** appliquons la règle de Bayes pour calculer la probabilité a posteriori. Au lieu de regarder uniquement la vrairessemblance, nous ajoutons la probabilité a priori de chaque classe. Cela devrait favoriser la classe majoritaire (IDL), car à la Q3 IDL est environ de 70% et NEG 30%, et réduire les fausses détections de colère. Cependant, comme nos scores de vraisemblance sont très grands, l'ajout du prior (qui est petit) risque d'avoir un impact limité sur la décision finale.

**Q9** Evaluer votre modèle en calculant le taux de bonnes réponses (accuracy) obtenu sur l'ensemble de test, c'est-à-dire le nombre de bonnes réponses sur le nombre total de segments de test. Vous deverez trouver un score légèrement entre 30 et 50%.

In [ ]:
# On compare notre prédiction Q8 avec la vraie réponse (y_test)
nb_correct = np.sum(predictions == y_test_np)
accuracy_Q9 = nb_correct / len(y_test_np)

print(f"Taux de bonnes réponses : {accuracy_Q9*100:.2f}%")


**Justification de Q9** Le taux de réussite est 36.92% en comparant à la colonne Emo initiale du segment test, il est faible, cela montre la limite au modèle naïf, car ce modèle considère les variables sont indépendantes mais nos données sont dépendantes car elles obtiennent par les vraies voix.

## 4) Comparaison de vos résultats avec sklearn (4 pts)

Enfin, je vous propose de comparer votre modèle avec une version implémentée dans la librairie `sklearn`. Plusieurs types de modèles sont possibles à choisir en fonction de vos données et votre problème: `MultinomialNB`, `GaussianNB`, `BernoulliNB`, etc...

`sklearn` fourni également un certain nombre de métriques très utiles en machine learning. Parmi celle-ci, la plus complète et adaptée aux problèmes de classification est la visualisation de la matrice de confusion. Cette matrice ($2 \times 2$ dans notre cas), permet de voir le nombre de segment correctement classés par classe.

In [ ]:
from sklearn.naive_bayes import MultinomialNB, GaussianNB, BernoulliNB
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay,  accuracy_score

L'apprentissage d'un modèle se fait avec les variables suivantes:
* `x_train`: matrice correspondant aux données (en lignes) et leurs descripteurs (en colonnes) pour **l'apprentissage** du modèle
* `y_train`: labels correspondant aux classes associés à chaque donnée d'apprentissage (vecteur)
* `x_test`: matrice correspondant aux données (en lignes) et leurs descripteurs (en colonnes) pour **l'évaluation** du modèle
* `y_pred`: labels prédits par le modèle sur les données d'évaluation `x_test`
* `y_test`: labels a priori inconnus correspondant aux classes associés à chaque donnée d'évaluation (vecteur)

**Q10** Apprendre un modèle équivalent à celui que vous avez implémenté en partie 3. Comparer les résultats obtenus avec les deux versions.

In [ ]:
### TO COMPLETE
# Calculer moyenne et écart-type sur les données de train
mean_train = np.mean(x_train_np, axis=0)
std_train = np.std(x_train_np, axis=0)
# Évite la division par 0
#supposon que
std_train[std_train == 0] = 1.0 

# Données normalisées
x_train_norm = (x_train_np - mean_train) / std_train
x_test_norm = (x_test_np - mean_train) / std_train

# Choix de modèle
# car selon les questions précedents, on ne travaille que la loi normale 
clf = GaussianNB() 

# Apprentissage
# Le modèle apprend tout seul les moyennes et variances comme Q3/Q4
clf.fit(x_train_norm, y_train)

# Prédiction
# Il calcule les probas comme et il décide
y_pred = clf.predict(x_test_norm)

# affichage la matrice de confusion
print(f"Taux de bonnes réponse : {clf.score(x_test_norm, y_test)*100:.2f}%")

cm = confusion_matrix(y_test, y_pred, labels=clf.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=clf.classes_)
disp.plot()
plt.show()

**Justification de Q10** On choissit le model GussienNB car les données de descripteurs suivent la loi normale, on va ensuite les amèner à la loi normale centrée reduite pour pemettre d'utiliser ce model car les données de chaque descripteurs sont heterogenes entre eux. Ce qui ramene la moyenne à 0 et l'écart-type entre -1 et 1.
En appliquant ce model de GaussienNB avec les mêmes données d'apprentissage, il repose sur même hypothèse de la partie 3.
On obtient le resultat de 36.91% de taux de réussite. Ce qui est extrement proche de premiere version 36.92% dans la partie 3, on peut donc dire que le calcul de moyenne, d'écart-type et de vraisemblance à la partie 3 sont correctes, c'est à dire, il n'y a pas erreurs dans cette partie, mais le modèle est toujours un peu médiocre.

**Q11** Le modèle ne donne pas de très bons résultats. Qu'est-ce qui pose problème à votre avis, quels sont les éléments qui peuvent être améliorés ?

Le probleme provient de descripteurs considères comme independants dans l'hypothese de Naïf car en vraie vie, tous les descripteur dune voix dependent de l'un à l'autre. En nos données de aibo sont les enregistrements de vrai environnement.

Un autre probleme est la modelisation par gaussien qui combine et modelise la classe NEG comme une seul moyenne et variance, vu que cettes classe est variée, donc le modele englobe tout ensemble et forme une zone de variance trop large. Cela absorbe par erreur la majorité des exemples de la classe 'Neutre'.


Pour ameliorer, 
on va reduire la dimension sur nos descripteurs qui possedent presque les memes donnnees, cela baisse le nombre de variables et elimine peu a peu la notion de dependance entre les descripteurs, ce qui gene l'hypothese de naif.

On peut faire probablement de remplace une seule modelisation par Gaussien par plusieurs fois de modelisation guassien, ça permet de modéliser séparément les sous-types d'émotions négatives et d'éviter que la variance ne devienne trop grande. C'est modèle GMM: https://scikit-learn.org/stable/modules/mixture.html


## 4) pour aller plus loin (5 pts)

Dans cette dernière partie, je vous laisse libre de choisir la tâche que vous souhaitez réaliser.
Finalement, vous pouvez rafiner votre modèle avec une approche de votre choix, faire une classification à plus que deux classes en utilisant le fichier de correspondance fourni. Vous pouvez également comparer l'approche avec d'autres types de modèles disponibles dans `sklearn`. Vous pouvez également prendre en compte le genre des locuteurs, ou normaliser vos descripteurs (par exemple pour que les descripteurs soient centrés réduits).

Vous n'aurez pas accès aux données audio directement. Par contre vous pouvez tester votre modèle sur Emo-DB un autre corpus (en allemand également) beaucoup plus petit et accessible sur la plateforme Kaggle (https://www.kaggle.com/datasets/piyushagni5/berlin-database-of-emotional-speech-emodb).

Bon courage!

In [ ]:
### TO COMPLETE
from sklearn.svm import SVC
from sklearn.mixture import GaussianMixture

# Séparer les données d'entraînement par classe
# On utilise les données normalisées de la Q10
x_train_idl = x_train_norm[y_train == 'IDL']
x_train_neg = x_train_norm[y_train == 'NEG']

# Créer et entraîner les modèles GMM
# n_components=5 veut dire qu'on utilise 5 "bosses" pour décrire l'émotion
# C'est beaucoup plus précis qu'une seule
# Plus de grand nombre de bosses utilisés, plus la prediction est précise => bon taux de bonnes réponses
# Mais avec Basthon je peux augmenter ce nombre, il va planté le PC
gmm_idl = GaussianMixture(n_components=5, covariance_type='full', random_state=42)
gmm_neg = GaussianMixture(n_components=5, covariance_type='full', random_state=42)
# Entraine
gmm_idl.fit(x_train_idl)
gmm_neg.fit(x_train_neg)

# Fonction pour prédire
def predict_gmm(x_data):
    # score_samples qui donne le vraisemblance
    log_likelihood_idl = gmm_idl.score_samples(x_data)
    log_likelihood_neg = gmm_neg.score_samples(x_data)
    
    # On ajoute les Priors
    # p_idl et p_neg viennent de calculs précédents
    final_score_idl = log_likelihood_idl + np.log(p_idl)
    final_score_neg = log_likelihood_neg + np.log(p_neg)
    
    return np.where(final_score_idl > final_score_neg, 'IDL', 'NEG')

# Tester sur l'ensemble de test
preds = predict_gmm(x_test_norm)
acc = np.mean(preds == y_test)
print(f"Taux de réussite avec GMM (5 composants) : {acc*100:.2f}%")
 
# Modèle : SVM
# Il cherche une frontière complexe entre les classes
clf_svm = SVC(kernel='rbf', C=1.0)
clf_svm.fit(x_train_norm, y_train)
acc_svm = accuracy_score(y_test, clf_svm.predict(x_test_norm))
print(f"Taux de réussite avec SVM : {acc_svm*100:.2f}%")

Pour donner la sythèse de ces taux: 

Modèle Mélange de Gaussiennes GMM modèlise sur 5 bosses ou sous classes permet d'augmenter le taux de bonnes réponses un peu près 58.61% en comparant au GussienNB avec 36.91%. Ce qui rapproche aux données attendues.

Encore plus meilleur, avec Support Vector Machine SVM: qui se contente de chercher la meilleure "frontière" mathématique pour séparer les deux classes et il rend les resultats plus efficaces.

En effet, les modèles génératifs comme GMM/Bayes ils essaient de dessiner tout le nuage de points, ce qui est très dur. Les modèles discriminants (SVM) essaient justement de construire un mur entre les deux nuages.

Sur des données brutes comme l'audio, il est plus facile de trouver la frontière que de dessiner correctement le nuage. Le SVM correspond donc bien à la classification d'audio.

In [ ]:
test_path = '/home/chaosok/Documents/L3/DM_math_python/'

test_train_dir = test_path + 'test/'
test_labels    = test_path + 'fichiers_suplementaires_question4/chunk_labels_2cl_corpus.txt' # à changer si vous voulez 5 classes d'émotions

test = pd.read_csv(test_labels, sep=' ',header=None, names=['name', 'emo2cl', 'val'])
del test['val']
test.set_index('name')
test

In [ ]:
#test_train = map_label_wav(test_train_dir, test)
#test_train
#spk_list_test = ['03']
#write_extract_features(test_train_dir, spk_list_test, 'test_train.csv')